<a href="https://colab.research.google.com/github/Chosencodes/Medical-Imaging-Projects/blob/main/Atrium_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm.notebook import tqdm
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [ ]:
sample_path = next(root.glob("la*.nii.gz"))

In [ ]:
sample_path_label = label / sample_path.name

In [ ]:
sample_path, sample_path_label

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_path_label)

mri = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
mri.shape,mask.shape

In [ ]:
nib.aff2axcodes(data.affine)

In [ ]:
!pip install celluloid -q
!apt-get install -y ffmpeg -q

In [ ]:
slice_num = 40

plt.figure(figsize=(6,6))
plt.imshow(mri[:,:,slice_num], cmap="bone")

mask_ = np.ma.masked_where(mask[:,:,slice_num] == 0,
                           mask[:,:,slice_num])

plt.imshow(mask_, alpha=0.5, cmap="autumn")

plt.show()

# **Preprocessing and Creating the Dataset**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [17]:
import torchio as tio
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [13]:
root = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/imagesTr")
label = Path("/content/drive/MyDrive/Atrium-Segmentation/Task02_Heart/labelsTr")

In [14]:
image_path = sorted(root.glob("la*.nii.gz"))
label_path = sorted(root.glob("la*.nii.gz"))

In [19]:
def make_subjects(image_path, label_path):
  subjects = []

  for img_path,lbl_path in zip(image_path, label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        label = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subjects = make_subjects(image_path, label_path)

In [21]:
train_subject,val_subject=train_test_split(all_subjects,test_size=0.15,random_state=13)

In [24]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.ZNormalization(),
    tio.RescaleIntensity(0,1),
    tio.CropOrPad((208,208,80),mask_name="label")
])

In [25]:
train_transform=tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0, 1, 2), flip_probability=0.5),
    tio.RandomAffine(scales=0.05, degrees=10, translation=5),
    tio.RandomNoise(mean=0, std=0.02),
])


val_transform=tio.Compose([
    base_transform
])

In [26]:
train_dataset = tio.SubjectsDataset(train_subject,transform=train_transform)
val_dataset = tio.SubjectsDataset(val_subject,transform=val_transform)